In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
!pip install ultralytics

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 58.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 123.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 99.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 59.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 4.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 42.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 20.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 5.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 104.3 MB/s eta 0:00:00
  Attempting uninstall: nvidia-nvjitlink-cu12
    Found existing installation: nvidia-nvjitlink-cu12 12.5.82
    Uninstalli

In [3]:
!unzip -q "/content/drive/MyDrive/yolo_file5.zip" -d /content/yolo_raw

In [4]:
import os, shutil

raw_base = "/content/yolo_raw"
yolo_base = "/content/yolo_data"

for split in ["train", "val", "test"]:
    os.makedirs(f"{yolo_base}/images/{split}", exist_ok=True)
    os.makedirs(f"{yolo_base}/labels/{split}", exist_ok=True)

    # Copy images
    for file in os.listdir(f"{raw_base}/{split}/images"):
        shutil.copy(f"{raw_base}/{split}/images/{file}", f"{yolo_base}/images/{split}/{file}")

    # Copy YOLO labels
    for file in os.listdir(f"{raw_base}/{split}/yolo_labels"):
        shutil.copy(f"{raw_base}/{split}/yolo_labels/{file}", f"{yolo_base}/labels/{split}/{file}")

In [5]:
yaml_path = "/content/yolo_data/dataset.yaml"

new_yaml = """\
path: /content/yolo_data
train: images/train
val: images/val
test: images/test

names:
  0: ChartTitle
  1: PlotArea
  2: LegendLabel
  3: xAxisLabel
  4: yAxisLabel
  5: PieLabel
  6: v_bar
  7: h_bar
  8: line
  9: yAxisTitle
"""

with open(yaml_path, "w") as f:
    f.write(new_yaml)

print(" dataset.yaml has been updated to correct format.")

 dataset.yaml has been updated to correct format.


In [6]:
for split in ['train', 'val', 'test']:
    empty = 0
    split_dir = f"/content/yolo_data/labels/{split}"
    for fname in os.listdir(split_dir):
        with open(os.path.join(split_dir, fname)) as f:
            if not f.read().strip():
                empty += 1
    print(f"{split}: {empty} empty label files")

train: 396 empty label files
val: 78 empty label files
test: 91 empty label files


In [7]:
import os

def clean_empty_labels(base_path):
    for split in ["train", "val", "test"]:
        folder = f"{base_path}/labels/{split}"
        removed = 0
        for file in os.listdir(folder):
            path = os.path.join(folder, file)
            if os.path.getsize(path) == 0:
                os.remove(path)
                removed += 1
        print(f"{split}: Removed {removed} empty label files.")

clean_empty_labels("/content/yolo_data")

train: Removed 396 empty label files.
val: Removed 78 empty label files.
test: Removed 91 empty label files.


In [8]:
cat /content/yolo_data/dataset.yaml

path: /content/yolo_data
train: images/train
val: images/val
test: images/test

names:
  0: ChartTitle
  1: PlotArea
  2: LegendLabel
  3: xAxisLabel
  4: yAxisLabel
  5: PieLabel
  6: v_bar
  7: h_bar
  8: line
  9: yAxisTitle


In [9]:
!pip install -q ultralytics

In [15]:
from ultralytics import YOLO

model = YOLO("yolov8n.pt")  # Change to yolov8s.pt or yolov8m.pt if needed

model.train(
    data="/content/yolo_data/dataset.yaml",
    epochs=50,
    imgsz=640,
    batch=16,
    name="chart_yolo"
)

Ultralytics 8.3.168 🚀 Python-3.11.13 torch-2.6.0+cu124 CUDA:0 (NVIDIA L4, 22693MiB)
engine/trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/yolo_data/dataset.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=50, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n.pt, momentum=0.937, mosaic=1.0, multi_scale=False, name=chart_yolo2, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, overlap_mask=True, patience=100, perspective=0.0, plots=True, pose=12.0, 

train: Scanning /content/yolo_data/labels/train.cache... 12425 images, 396 backgrounds, 0 corrupt: 100%|██████████| 12821/12821 [00:00<?, ?it/s]

train: /content/yolo_data/images/train/04769019006554.png: 1 duplicate labels removed
train: /content/yolo_data/images/train/08995001006669.png: 7 duplicate labels removed
train: /content/yolo_data/images/train/10669853002939.png: 1 duplicate labels removed
train: /content/yolo_data/images/train/10912.png: 1 duplicate labels removed
train: /content/yolo_data/images/train/11428.png: 1 duplicate labels removed
train: /content/yolo_data/images/train/12217.png: 3 duplicate labels removed
train: /content/yolo_data/images/train/1631.png: 4 duplicate labels removed
train: /content/yolo_data/images/train/2606.png: 3 duplicate labels removed
train: /content/yolo_data/images/train/3236.png: 5 duplicate labels removed
train: /content/yolo_data/images/train/4088.png: 3 duplicate labels removed
train: /content/yolo_data/images/train/6056.png: 2 duplicate labels removed
train: /content/yolo_data/images/train/7893.png: 1 duplicate labels removed
train: /content/yolo_data/images/train/8298.png: 2 dupl

val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 504.3±298.0 MB/s, size: 46.0 KB)


val: Scanning /content/yolo_data/labels/val.cache... 2670 images, 78 backgrounds, 0 corrupt: 100%|██████████| 2748/2748 [00:00<?, ?it/s]

...

      50/50      7.53G     0.4329     0.3202     0.8072        132        640: 100%|██████████| 802/802 [01:23<00:00,  9.61it/s]
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 86/86 [00:13<00:00,  6.46it/s]


                   all       2748      94701      0.942      0.941      0.952      0.838

50 epochs completed in 1.483 hours.
Optimizer stripped from runs/detect/chart_yolo2/weights/last.pt, 6.2MB
Optimizer stripped from runs/detect/chart_yolo2/weights/best.pt, 6.2MB

Validating runs/detect/chart_yolo2/weights/best.pt...
Ultralytics 8.3.168 🚀 Python-3.11.13 torch-2.6.0+cu124 CUDA:0 (NVIDIA L4, 22693MiB)
Model summary (fused): 72 layers, 3,007,598 parameters, 0 gradients, 8.1 GFLOPs


                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 86/86 [00:16<00:00,  5.07it/s]


                   all       2748      94701      0.942      0.941      0.952      0.838
            ChartTitle        248        248      0.974      0.988      0.994        0.9
              PlotArea       2625       2625      0.969      0.998      0.989      0.987
           LegendLabel        570       1743      0.943      0.953      0.956        0.8
            xAxisLabel       2588      29463      0.954      0.975      0.981       0.88
            yAxisLabel       2461      18978      0.937      0.958      0.968      0.859
              PieLabel         81        353      0.916      0.943      0.954       0.68
                 v_bar       1511      19791      0.982      0.936      0.958      0.917
                 h_bar        752      11387      0.896      0.904      0.887      0.806
                  line        322       7735      0.934      0.773      0.862      0.703
            yAxisTitle       2378       2378      0.918      0.977      0.974      0.847
Speed: 0.1ms preproce

ultralytics.utils.metrics.DetMetrics object with attributes:

ap_class_index: array([0, 1, 2, 3, 4, 5, 6, 7, 8, 9])
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x7c76167a8290>
curves: ['Precision-Recall(B)', 'F1-Confidence(B)', 'Precision-Confidence(B)', 'Recall-Confidence(B)']
curves_results: [[array([          0,    0.001001,    0.002002,    0.003003,    0.004004,    0.005005,    0.006006,    0.007007,    0.008008,    0.009009,     0.01001,    0.011011,    0.012012,    0.013013,    0.014014,    0.015015,    0.016016,    0.017017,    0.018018,    0.019019,     0.02002,    0.021021,    0.022022,    0.023023,
          0.024024,    0.025025,    0.026026,    0.027027,    0.028028,    0.029029,     0.03003,    0.031031,    0.032032,    0.033033,    0.034034,    0.035035,    0.036036,    0.037037,    0.038038,    0.039039,     0.04004,    0.041041,    0.042042,    0.043043,    0.044044,    0.045045,    0.046046,    0.0

In [16]:
!ls runs/detect/chart_yolo*/weights/

runs/detect/chart_yolo2/weights/:
best.pt  last.pt

runs/detect/chart_yolo/weights/:
best.pt  last.pt


In [17]:
from ultralytics import YOLO

# Load your best model
model = YOLO('runs/detect/chart_yolo2/weights/best.pt')  # or chart_yolo4 if preferred

# Evaluate on the test split
metrics = model.val(split='val')


Ultralytics 8.3.168 🚀 Python-3.11.13 torch-2.6.0+cu124 CUDA:0 (NVIDIA L4, 22693MiB)
Model summary (fused): 72 layers, 3,007,598 parameters, 0 gradients, 8.1 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 1425.5±512.5 MB/s, size: 38.9 KB)


val: Scanning /content/yolo_data/labels/val.cache... 2670 images, 78 backgrounds, 0 corrupt: 100%|██████████| 2748/2748 [00:00<?, ?it/s]

val: /content/yolo_data/images/val/6912.png: 3 duplicate labels removed
val: /content/yolo_data/images/val/multi_col_142.png: 1 duplicate labels removed
val: /content/yolo_data/images/val/multi_col_20254.png: 2 duplicate labels removed
val: /content/yolo_data/images/val/multi_col_213.png: 1 duplicate labels removed
val: /content/yolo_data/images/val/multi_col_40564.png: 1 duplicate labels removed
val: /content/yolo_data/images/val/multi_col_498.png: 1 duplicate labels removed
val: /content/yolo_data/images/val/multi_col_60687.png: 1 duplicate labels removed



                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 172/172 [00:18<00:00,  9.55it/s]


                   all       2748      94701      0.941      0.941      0.952      0.844
            ChartTitle        248        248      0.976      0.985      0.995      0.899
              PlotArea       2625       2625      0.969      0.998      0.989      0.987
           LegendLabel        570       1743      0.942      0.952      0.955      0.817
            xAxisLabel       2588      29463      0.953      0.975      0.981      0.895
            yAxisLabel       2461      18978      0.936      0.959      0.968      0.863
              PieLabel         81        353      0.912      0.942      0.953      0.684
                 v_bar       1511      19791      0.981      0.937      0.958      0.921
                 h_bar        752      11387      0.894      0.906      0.887       0.81
                  line        322       7735      0.932      0.776      0.861      0.709
            yAxisTitle       2378       2378      0.919      0.978      0.973      0.852
Speed: 0.2ms preproce

In [18]:
from ultralytics import YOLO

# Load the trained model
model = YOLO('runs/detect/chart_yolo2/weights/best.pt')

# Evaluate on the test set
metrics = model.val(split='test')


Ultralytics 8.3.168 🚀 Python-3.11.13 torch-2.6.0+cu124 CUDA:0 (NVIDIA L4, 22693MiB)
Model summary (fused): 72 layers, 3,007,598 parameters, 0 gradients, 8.1 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 1243.2±432.8 MB/s, size: 36.4 KB)


val: Scanning /content/yolo_data/labels/test.cache... 2657 images, 91 backgrounds, 0 corrupt: 100%|██████████| 2748/2748 [00:00<?, ?it/s]

val: /content/yolo_data/images/test/1336.png: 3 duplicate labels removed
val: /content/yolo_data/images/test/2191.png: 1 duplicate labels removed
val: /content/yolo_data/images/test/2748.png: 3 duplicate labels removed
val: /content/yolo_data/images/test/3958.png: 2 duplicate labels removed
val: /content/yolo_data/images/test/multi_col_80510.png: 2 duplicate labels removed
val: /content/yolo_data/images/test/multi_col_80758.png: 3 duplicate labels removed



                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 172/172 [00:18<00:00,  9.52it/s]


                   all       2748      92432      0.936      0.942      0.951      0.844
            ChartTitle        274        274      0.982      0.994      0.994      0.886
              PlotArea       2618       2618      0.967      0.998       0.99      0.988
           LegendLabel        581       1753       0.95      0.947      0.948      0.818
            xAxisLabel       2579      28647      0.945      0.978      0.982      0.899
            yAxisLabel       2443      18761      0.933       0.96      0.971      0.867
              PieLabel         78        352      0.868      0.935      0.934       0.67
                 v_bar       1514      19589      0.989      0.952      0.971      0.937
                 h_bar        742      10861      0.865      0.894      0.868      0.799
                  line        323       7235      0.942      0.782      0.875      0.727
            yAxisTitle       2342       2342      0.922      0.983      0.972      0.847
Speed: 0.2ms preproce

In [20]:
!cp /content/runs/detect/chart_yolo2/weights/best.pt /content/drive/MyDrive/yolo_model_e50/chart_yolo2_best.pt
